# Membangun KNN dari Nol — Data berupa *List of Dictionaries*

Di notebook ini kita membangun sendiri sebuah K-Nearest Neighbors classifier, dengan dua aturan:

> **1. API-nya harus terasa persis seperti scikit-learn** — ada `.fit(X, y)` dan `.predict(X)`.
> **2. Data masuk berupa `list of dict`** — bukan `list of list`, bukan `DataFrame`.

Aturan kedua itu bukan dibuat-buat. Begitulah bentuk data di dunia nyata:

- Respons **API** dan file **JSON** hampir selalu berupa daftar record.
- **`df.to_dict("records")`** di pandas menghasilkan bentuk ini.
- Baris hasil query **database** (`DictCursor`) juga.
- Satu baris **CSV** yang dibaca `csv.DictReader` juga.

Perbedaannya dengan `list of list` terlihat sepele — sekadar ganti `[1.0, 2.0]` jadi
`{"x": 1.0, "y": 2.0}`. Ternyata tidak. Dict punya **kunci bernama tanpa urutan yang dijamin**,
dan itu memunculkan satu kelas bug yang sama sekali tidak ada pada list.
Menemukan dan menutup bug itu adalah separuh isi notebook ini.

---

### Peta perjalanan

**BAGIAN 1 — Versi `list of dict`**
Input `list of dict`, output `list`. Python murni, tanpa numpy. Kita akan menemukan
jebakan urutan kunci lebih dulu, baru membangun class-nya.

**BAGIAN 2 — Versi `ndarray`**
Kita jembatani dunia dict ke dunia numpy lewat satu fungsi konversi, lalu vektorkan semuanya.
Input tetap `list of dict`, output `ndarray`.

**BAGIAN 3 — Verifikasi & benchmark**
Adu hasil dengan `KNeighborsClassifier()` dan `DictVectorizer()` asli, lalu ukur waktunya.

## 0. Kontrak scikit-learn: apa yang kita tiru?

| Method | Input | Output | Tugasnya |
|---|---|---|---|
| `__init__(n_neighbors)` | hyperparameter | — | Simpan setelan. **Tidak menyentuh data.** |
| `fit(X, y)` | data training | `self` | Belajar dari data. |
| `predict(X)` | data baru | prediksi label | Menebak. |

Dua aturan resmi scikit-learn yang sering dilewatkan:

1. **`__init__` tidak boleh melihat data.** Ia hanya menyimpan hyperparameter. Inilah yang membuat
   `GridSearchCV` bisa meng-*clone* model berkali-kali tanpa efek samping.
2. **`fit` mengembalikan `self`.** Itu sebabnya `model.fit(X, y).predict(X_baru)` bisa dirangkai.

### Keanehan khusus KNN

Kebanyakan model bekerja keras di `fit`. KNN kebalikannya — ia **lazy learner**:

- `fit` → cuma **menyimpan** data training. Nyaris tanpa komputasi.
- `predict` → di sinilah **semua** kerja beratnya terjadi.

### Satu tugas tambahan karena kita memakai dict

Ada satu hal yang wajib dipelajari `fit` kita, yang tidak diperlukan versi `list of list`:

> **Fitur apa saja yang ada, dan dalam urutan apa?**

Pada `list of list`, urutan fitur sudah ditentukan oleh posisi kolom — kolom ke-0 selalu kolom ke-0.
Pada `list of dict`, tidak ada "posisi". Yang ada hanya nama. Jadi model kita harus **menetapkan
sendiri** urutan kanonik itu saat `fit`, lalu memakainya secara konsisten selamanya.

Atribut hasil belajar ini akan kita simpan sebagai `feature_names_` — persis nama yang dipakai
scikit-learn.

---
# BAGIAN 1 — Versi `list of dict`

Aturan bagian ini: **tanpa numpy sama sekali.** Input `list of dict`, output `list`.

## 1.1 Dataset mainan

Dua kelompok titik di bidang 2D: satu berkerumun di kiri-bawah (kelas `0`), satu di kanan-atas
(kelas `1`). Sekarang tiap titik adalah sebuah dict dengan kunci `"x"` dan `"y"`.

In [ ]:
X_train = [
    {"x": 1.0, "y": 1.0},   # kelas 0
    {"x": 1.5, "y": 2.0},   # kelas 0
    {"x": 2.0, "y": 1.5},   # kelas 0
    {"x": 1.2, "y": 1.8},   # kelas 0
    {"x": 5.0, "y": 5.0},   # kelas 1
    {"x": 5.5, "y": 4.5},   # kelas 1
    {"x": 6.0, "y": 5.2},   # kelas 1
    {"x": 5.2, "y": 5.8},   # kelas 1
]
y_train = [0, 0, 0, 0, 1, 1, 1, 1]

X_test = [
    {"x": 1.4, "y": 1.4},   # jelas dekat kelompok kiri-bawah -> harusnya 0
    {"y": 5.1, "x": 5.4},   # jelas dekat kanan-atas -> harusnya 1   (perhatikan urutan kuncinya!)
    {"x": 3.5, "y": 3.5},   # tepat di tengah -> menarik
]

print("Jumlah data training :", len(X_train))
print("Satu record          :", X_train[0])
print("Tipe record          :", type(X_train[0]))
print("Kunci yang tersedia  :", list(X_train[0].keys()))

Perhatikan `X_test[1]`. Kuncinya ditulis `{"y": ..., "x": ...}` — terbalik dari yang lain.

Ini **bukan salah ketik**. Ini sengaja, dan mewakili kenyataan: JSON tidak menjamin urutan kunci,
dan dua panggilan API yang sama bisa mengembalikan record dengan urutan berbeda. Kalau kode kita
peka terhadap urutan penulisan, ia akan rusak justru pada data yang secara isi sepenuhnya benar.

Mari kita buktikan bahwa Python sendiri menganggap urutan itu tidak penting:

In [ ]:
a = {"x": 1.0, "y": 5.0}
b = {"y": 5.0, "x": 1.0}      # data identik, urutan penulisan berbeda

print("a       :", a)
print("b       :", b)
print("a == b ?:", a == b, "  <- Python bilang keduanya SAMA")
print()
print("Tapi urutan nilainya berbeda:")
print("  list(a.values()) =", list(a.values()))
print("  list(b.values()) =", list(b.values()))

Baris `a == b` bernilai `True`: bagi Python, dict dibandingkan berdasarkan **isi**, bukan urutan.
Tapi `.values()` mengembalikan nilai sesuai **urutan penyisipan**, dan urutan itu berbeda.

Dua fakta ini tidak akur. Di sinilah bug kita akan lahir.

## 1.2 Jebakan pertama — jangan pernah pakai `.values()`

Rumus jarak Euclidean tetap sama seperti biasa:

$$d(p, q) = \sqrt{\sum_{i} (p_i - q_i)^2}$$

Terjemahan paling "alami" ke dict biasanya begini — pasangkan nilai-nilainya dengan `zip`.
Mari kita lihat apa yang terjadi.

In [ ]:
import math

def jarak_naif(p, q):
    '''VERSI SALAH. Jangan dipakai. Ada di sini untuk dibongkar.'''
    total = 0.0
    for nilai_p, nilai_q in zip(p.values(), q.values()):
        total += (nilai_p - nilai_q) ** 2
    return math.sqrt(total)


print("a == b ?          :", a == b)
print("Jarak seharusnya  : 0.0")
print("jarak_naif(a, b)  :", jarak_naif(a, b))
print()
print("Pasangan yang dibentuk zip:", list(zip(a.values(), b.values())))

Jaraknya **5.657**, padahal seharusnya **0.0**. Dua record yang Python sendiri nyatakan identik,
menurut fungsi kita berjarak sangat jauh.

Penyebabnya terlihat di baris terakhir: `zip` memasangkan nilai berdasarkan **posisi**, sehingga
`x` milik `a` dipasangkan dengan `y` milik `b`. Kita membandingkan apel dengan jeruk, lalu
mengkuadratkannya dengan penuh percaya diri.

Yang membuat bug ini berbahaya bukan besarnya kesalahan, melainkan **diamnya**:

- Tidak ada `Exception`. Tidak ada peringatan.
- Hasilnya tetap `float` yang tampak wajar.
- Model tetap jalan, tetap memberi prediksi, akurasinya cuma "agak jelek".
- Dan ia hanya muncul pada **sebagian** record — yaitu record yang kebetulan urutan kuncinya
  berbeda. Jadi bug ini tidak bisa direproduksi secara konsisten.

Bug yang berteriak itu mudah. Bug yang berbisik seperti ini yang menghabiskan waktu berhari-hari.

## 1.3 Perbaikannya — kunci sebagai sumber kebenaran

Solusinya sederhana: **jangan pernah mengandalkan urutan.** Akses nilai lewat namanya,
dan tetapkan satu daftar fitur kanonik yang dipakai oleh seluruh perhitungan.

In [ ]:
def euclidean_distance(p, q, feature_names):
    '''
    Jarak Euclidean antara dua record dict.

    p, q          : dict  -- dua record
    feature_names : list  -- daftar fitur kanonik, menentukan APA yang dibandingkan
                             sekaligus memastikan kedua record dibaca dengan cara sama
    return        : float
    '''
    total = 0.0
    for nama in feature_names:
        selisih = p[nama] - q[nama]    # akses berdasarkan NAMA, bukan posisi
        total += selisih ** 2
    return math.sqrt(total)


fitur = ["x", "y"]

print("euclidean_distance(a, b) :", euclidean_distance(a, b, fitur), " <- benar")
print()
# Uji dengan segitiga 3-4-5 yang jawabannya sudah kita tahu
p1 = {"x": 0.0, "y": 0.0}
p2 = {"y": 4.0, "x": 3.0}          # sengaja urutannya dibalik
print("Jarak (0,0) ke (3,4)     :", euclidean_distance(p1, p2, fitur), " <- harus 5.0")

Sekarang urutan penulisan kunci tidak berpengaruh sama sekali, karena kita tidak pernah bertanya
"nilai ke berapa?" — kita selalu bertanya "nilai bernama apa?".

Sekalian mari kita periksa apakah rumusnya masih rumus yang sama. Setiap potongannya ada untuk
memperbaiki masalah yang dibuat potongan sebelumnya:

| Potongan | Kenapa ada di situ? |
|---|---|
| `feature_names` | Menjamin kedua record dibaca dengan urutan dan himpunan fitur yang sama. **Ini bagian baru, khusus dict.** |
| $p_i - q_i$ | Selisih per fitur. Tapi bisa negatif. |
| $(\;)^2$ | Menghilangkan tanda negatif; selisih besar dihukum lebih berat. |
| $\sum$ | Gabungkan semua fitur jadi satu angka. |
| $\sqrt{\;}$ | Kembalikan ke satuan asli data. |

> **Catatan.** $\sqrt{\cdot}$ sebenarnya tidak wajib kalau kita hanya ingin *mengurutkan* jarak,
> sebab akar adalah fungsi monoton naik. Kita tetap memakainya supaya angkanya terbaca sebagai
> jarak sungguhan.

## 1.4 Jebakan kedua — kunci yang hilang

Jebakan urutan sudah tertutup. Masih ada satu lagi yang khas dict: **record yang tidak lengkap.**

Pada `list of list`, baris pendek langsung terasa aneh — panjangnya jelas berbeda.
Pada dict, record yang kehilangan satu kunci terlihat baik-baik saja sampai kita menyentuhnya.

In [ ]:
record_rusak = {"x": 1.0}          # kunci "y" hilang

try:
    euclidean_distance(record_rusak, X_train[0], fitur)
except KeyError as e:
    print("KeyError:", e)
    print("\nPesan ini benar, tapi tidak membantu:")
    print("  - tidak menyebut record ke berapa")
    print("  - tidak menyebut fitur apa saja yang diharapkan")
    print("  - kalau ada 3 kunci hilang, ia cuma melaporkan satu")

Kita bisa jauh lebih baik dari itu. Kegagalan yang cepat dan informatif jauh lebih murah daripada
kegagalan yang membingungkan — apalagi dibandingkan kegagalan yang diam.

In [ ]:
def validasi_records(records, feature_names, nama_data="X"):
    '''Pastikan setiap record punya seluruh fitur yang dibutuhkan.'''
    for i, rec in enumerate(records):
        if not isinstance(rec, dict):
            raise TypeError(f"{nama_data}[{i}] bertipe {type(rec).__name__}, seharusnya dict")
        hilang = [f for f in feature_names if f not in rec]
        if hilang:
            raise ValueError(
                f"{nama_data}[{i}] kehilangan fitur {hilang}. "
                f"Diharapkan: {feature_names}. Yang ada: {sorted(rec.keys())}"
            )


try:
    validasi_records([{"x": 1.0, "y": 2.0}, {"x": 3.0}], fitur)
except ValueError as e:
    print("ValueError:", e)

Sekarang pesannya menyebut baris keberapa, fitur apa yang hilang, dan apa yang sebenarnya ada.

> **Kenapa tidak diisi 0 saja?** Karena untuk KNN, `0` **bukan** penanda "tidak ada" — ia adalah
> koordinat yang sah. Record `{"x": 1.0}` yang dilengkapi jadi `{"x": 1.0, "y": 0.0}` akan
> ditempatkan di titik `(1, 0)`, lalu ikut dihitung jaraknya seolah-olah itu memang lokasinya.
> Modelnya tetap jalan, tetap memberi jawaban, dan tetap salah tanpa memberi tahu siapa pun.
>
> Kita akan lihat di Bagian 3 bahwa `DictVectorizer` milik scikit-learn justru melakukan hal itu —
> mengisi 0 secara diam-diam. Untuk kasus penggunaannya (data teks yang memang jarang/*sparse*),
> itu masuk akal. Untuk KNN dengan fitur numerik, itu berbahaya.

## 1.5 Langkah berikutnya — jarak ke *semua* titik training

Jebakan sudah lewat. Sisanya adalah KNN biasa.

In [ ]:
def semua_jarak(x_baru, X_train, feature_names):
    '''Jarak dari satu record test ke seluruh record training.'''
    return [euclidean_distance(x_baru, x_train, feature_names) for x_train in X_train]


x = X_test[0]
jarak = semua_jarak(x, X_train, fitur)

print(f"Titik test: {x}\n")
for i, d in enumerate(jarak):
    print(f"  index {i}  {str(X_train[i]):<28} label={y_train[i]}   jarak={d:.4f}")

Empat jarak pertama kecil, empat berikutnya besar — titik test ini memang di wilayah kelas `0`.

**Ongkosnya:** untuk $m$ titik test dan $n$ titik training, kita memanggil `euclidean_distance`
sebanyak $m \times n$ kali. Ingat angka ini; ia alasan utama kita pindah ke numpy di Bagian 2.

## 1.6 Ambil $k$ tetangga terdekat

Kita butuh **indeks** dari $k$ jarak terkecil — bukan nilainya, sebab lewat indeks itulah
kita bisa menengok label di `y_train`.

In [ ]:
def k_tetangga_terdekat(jarak, k):
    '''Indeks dari k jarak terkecil, terurut dari yang paling dekat.'''
    urutan = sorted(range(len(jarak)), key=lambda i: jarak[i])
    return urutan[:k]


k = 3
idx = k_tetangga_terdekat(jarak, k)

print(f"Indeks {k} tetangga terdekat: {idx}\n")
for i in idx:
    print(f"  index {i}  {str(X_train[i]):<28} label={y_train[i]}   jarak={jarak[i]:.4f}")

> **Ini pola `argsort`.** `sorted(range(n), key=...)` mengurutkan *indeks* berdasarkan *nilai*.
> Namanya "arg"-sort karena yang dikembalikan argumen (posisi), bukan nilainya.
> Ingat namanya — numpy punya `np.argsort()` yang melakukan hal sama, jauh lebih cepat.

## 1.7 Voting mayoritas — dan soal seri

Tinggal ambil label para tetangga, lalu pilih yang paling sering muncul.

In [ ]:
from collections import Counter

def voting_mayoritas(label_tetangga):
    '''Label terbanyak di antara tetangga.'''
    hitungan = Counter(label_tetangga)
    return hitungan.most_common(1)[0][0]


label_tetangga = [y_train[i] for i in idx]
print("Label para tetangga :", label_tetangga)
print("Hitungan suara      :", dict(Counter(label_tetangga)))
print("Pemenang            :", voting_mayoritas(label_tetangga))

In [ ]:
# Apa yang terjadi kalau suaranya imbang?
print("Label [0, 1] ->", voting_mayoritas([0, 1]))
print("Label [1, 0] ->", voting_mayoritas([1, 0]))

Perhatikan: **himpunan tetangganya identik**, hanya urutannya berbeda, tapi jawabannya berubah.
`Counter.most_common()` memutus seri berdasarkan urutan kemunculan pertama.

Ini pola yang sama persis dengan jebakan dict di awal tadi: **hasil yang bergantung pada urutan,
padahal urutan tidak bermakna.** Dan seperti sebelumnya, obatnya juga sama — buat aturannya eksplisit.

Kalau seri, menangkan label **terkecil**. Aturan ini sembarang, tapi *deterministik*, dan
kebetulan sama persis dengan yang akan dilakukan `argmax` numpy nanti.

In [ ]:
def voting_mayoritas(label_tetangga):
    '''
    Label terbanyak di antara tetangga.
    Kalau seri: menangkan label terkecil (deterministik).
    '''
    hitungan = Counter(label_tetangga)
    suara_maks = max(hitungan.values())
    kandidat = [label for label, n in hitungan.items() if n == suara_maks]
    return min(kandidat)


print("Label [0, 1] ->", voting_mayoritas([0, 1]))
print("Label [1, 0] ->", voting_mayoritas([1, 0]), " <- sekarang konsisten")
print("Label [1, 1, 2, 2, 0] ->", voting_mayoritas([1, 1, 2, 2, 0]), " (seri 1 vs 2 -> menang 1)")

> **Tetap pakai $k$ ganjil untuk masalah biner.** Aturan pemutus seri di atas adalah jaring
> pengaman, bukan pengganti desain yang baik. Dengan $k$ ganjil pada 2 kelas, seri suara mustahil
> terjadi sejak awal. Untuk 3 kelas atau lebih, $k$ ganjil **tidak** menjamin apa-apa.

## 1.8 Rakit jadi class

Sekarang kita bungkus dengan kontrak scikit-learn. Perhatikan empat hal:

1. `__init__` **hanya** menyimpan `n_neighbors`. Tidak menyentuh data.
2. `fit` **mempelajari `feature_names_`** dari kunci-kunci record — inilah tugas tambahan
   yang tidak ada di versi `list of list`.
3. `fit` menyalin data lalu `return self`. Itu saja. Wajah *lazy learning*.
4. `predict` **memvalidasi** setiap record masuk terhadap `feature_names_` hasil `fit`.

Konvensi penamaan: garis bawah di depan (`_predict_satu`) = method internal.
Garis bawah di belakang (`feature_names_`) = konvensi scikit-learn untuk atribut yang
**baru ada setelah `fit`**.

Soal urutan fitur, kita pakai **`sorted()`** atas gabungan seluruh kunci. Kenapa `sorted` dan bukan
urutan kemunculan? Karena `sorted` menghasilkan urutan yang sama untuk data yang sama, tidak peduli
record mana yang kebetulan datang duluan. Bonus: itu persis yang dilakukan `DictVectorizer`,
seperti akan kita buktikan di Bagian 3.

In [ ]:
class KNNClassifierDict:
    '''
    K-Nearest Neighbors untuk data list of dict. Python murni.

    Input  : list of dict (X), list (y)
    Output : list
    '''

    def __init__(self, n_neighbors=3):
        # HANYA hyperparameter. Tidak ada data di sini.
        self.n_neighbors = n_neighbors

    # ---------- FIT ----------
    def fit(self, X, y):
        if len(X) != len(y):
            raise ValueError(f"X ({len(X)} record) dan y ({len(y)} label) tidak sama panjang")
        if self.n_neighbors > len(X):
            raise ValueError(
                f"n_neighbors={self.n_neighbors} lebih besar dari jumlah data training ({len(X)})"
            )
        if len(X) == 0:
            raise ValueError("X kosong")

        # --- Belajar daftar fitur: gabungan semua kunci, diurutkan ---
        kunci = set()
        for rec in X:
            if not isinstance(rec, dict):
                raise TypeError(f"Semua elemen X harus dict, dapat {type(rec).__name__}")
            kunci.update(rec.keys())
        self.feature_names_ = sorted(kunci)

        # --- Pastikan tidak ada record yang tidak lengkap ---
        validasi_records(X, self.feature_names_, "X")

        # Salin datanya, jangan simpan referensinya, supaya perubahan di luar
        # tidak diam-diam mengubah isi model.
        self.X_train_ = [dict(rec) for rec in X]
        self.y_train_ = list(y)

        return self          # WAJIB: supaya .fit(...).predict(...) bisa dirangkai

    # ---------- HELPER ----------
    def _cek_sudah_fit(self):
        if not hasattr(self, "X_train_"):
            raise RuntimeError("Model belum di-fit. Panggil .fit(X, y) terlebih dahulu.")

    def _predict_satu(self, x):
        '''Prediksi untuk SATU record. Ini inti algoritmanya.'''
        jarak = [
            euclidean_distance(x, x_train, self.feature_names_)
            for x_train in self.X_train_
        ]
        idx = sorted(range(len(jarak)), key=lambda i: jarak[i])[:self.n_neighbors]
        label = [self.y_train_[i] for i in idx]
        return voting_mayoritas(label)

    # ---------- PREDICT ----------
    def predict(self, X):
        self._cek_sudah_fit()
        validasi_records(X, self.feature_names_, "X")     # tolak record cacat lebih awal
        return [self._predict_satu(x) for x in X]         # output: list

    # ---------- SCORE ----------
    def score(self, X, y):
        y_pred = self.predict(X)
        benar = sum(1 for a, b in zip(y_pred, y) if a == b)
        return benar / len(y)

In [ ]:
model = KNNClassifierDict(n_neighbors=3)
model.fit(X_train, y_train)

print("Fitur yang dipelajari :", model.feature_names_)
print()

prediksi = model.predict(X_test)
print("Prediksi    :", prediksi)
print("Tipe output :", type(prediksi))
print()
for x, p in zip(X_test, prediksi):
    print(f"  {str(x):<28} -> kelas {p}")

Titik kedua — yang kuncinya sengaja ditulis terbalik `{"y": 5.1, "x": 5.4}` — mendapat prediksi
yang benar. Itu bukti bahwa `feature_names_` sudah menjalankan tugasnya.

Mari kita uji lebih keras. Kita acak urutan kunci setiap record test, lalu pastikan prediksinya
tidak bergeser sedikit pun:

In [ ]:
import random

def acak_urutan_kunci(rec, seed):
    '''Bikin dict baru dengan isi sama tapi urutan penyisipan diacak.'''
    kunci = list(rec.keys())
    random.Random(seed).shuffle(kunci)
    return {k: rec[k] for k in kunci}


X_test_acak = [acak_urutan_kunci(r, seed=i) for i, r in enumerate(X_test)]

print("Sebelum diacak:", X_test)
print("Sesudah diacak:", X_test_acak)
print()
print("Isinya masih sama? ", X_test_acak == X_test)
print()
print("Prediksi asli  :", model.predict(X_test))
print("Prediksi acak  :", model.predict(X_test_acak))
print("Identik?       :", model.predict(X_test) == model.predict(X_test_acak))

In [ ]:
# Titik acuan dengan nilai x dan y yang BERBEDA -- ini penting, lihat catatan di bawah
acuan = X_train[5]                      # {"x": 5.5, "y": 4.5}

kembar = [
    {"x": 5.4, "y": 5.1},               # ditulis x dulu
    {"y": 5.1, "x": 5.4},               # data SAMA PERSIS, ditulis y dulu
]

print("Acuan          :", acuan)
print("Kedua record sama isinya?", kembar[0] == kembar[1])
print()
print(f"{'Record':<28}{'euclidean_distance':>20}{'jarak_naif':>14}")
print("-" * 62)
for r in kembar:
    print(f"{str(r):<28}{euclidean_distance(r, acuan, fitur):>20.4f}"
          f"{jarak_naif(r, acuan):>14.4f}")

Kolom tengah konsisten: **0.6083** untuk kedua record, sebagaimana mestinya — isinya memang sama.

Kolom kanan tidak. Record kedua mendapat **0.9849**, padahal ia record yang sama persis, hanya
ditulis dengan urutan kunci berbeda. `jarak_naif` memasangkan `y` milik record dengan `x` milik
acuan, lalu melaporkan angka yang tampak sepenuhnya wajar.

> **Kenapa titik acuannya harus punya nilai x dan y yang berbeda?**
> Coba ganti `acuan` menjadi `X_train[4]`, yaitu `{"x": 5.0, "y": 5.0}`, lalu jalankan ulang sel di atas.
> Kedua kolom akan cocok, dan bug-nya seolah hilang. Sebabnya: kalau $x = y$, tertukarnya pasangan
> tidak mengubah apa pun — $(a-5)^2 + (b-5)^2$ sama saja bagaimanapun urutannya.
>
> Ini justru memperkuat pesan utamanya. Bug urutan kunci **tidak muncul di semua data**.
> Ia bersembunyi di balik titik-titik simetris, lalu menyerang yang lain. Test case yang
> kebetulan simetris akan lulus dengan gemilang sambil membiarkan bug-nya lolos ke produksi.

In [ ]:
# Chaining: mungkin karena fit() mengembalikan self
print(KNNClassifierDict(n_neighbors=3).fit(X_train, y_train).predict(X_test))
print()

# Belum di-fit
try:
    KNNClassifierDict(n_neighbors=3).predict(X_test)
except RuntimeError as e:
    print("RuntimeError :", e)

# k terlalu besar
try:
    KNNClassifierDict(n_neighbors=99).fit(X_train, y_train)
except ValueError as e:
    print("ValueError   :", e)

# Record test kehilangan fitur
try:
    model.predict([{"x": 1.0}])
except ValueError as e:
    print("ValueError   :", e)

# Fitur asing ikut terbawa -> diabaikan, karena semua fitur wajib tetap ada
print("\nRecord dengan kunci asing:",
      model.predict([{"x": 1.4, "y": 1.4, "catatan": "diabaikan"}]))

In [ ]:
x_tengah = {"x": 3.5, "y": 3.5}
jarak_tengah = semua_jarak(x_tengah, X_train, model.feature_names_)
urut = sorted(range(len(jarak_tengah)), key=lambda i: jarak_tengah[i])

print(f"Semua tetangga {x_tengah}, diurutkan dari yang terdekat:\n")
for peringkat, i in enumerate(urut, start=1):
    tanda = "  <-- masuk k=3" if peringkat <= 3 else ""
    print(f"  #{peringkat}  {str(X_train[i]):<28} label={y_train[i]}   "
          f"jarak={jarak_tengah[i]:.4f}{tanda}")

In [ ]:
# Titik yang sama, k berbeda -> jawaban bisa berubah
for k_coba in [1, 3, 5, 7]:
    m = KNNClassifierDict(n_neighbors=k_coba).fit(X_train, y_train)
    print(f"k={k_coba}  ->  prediksi untuk {x_tengah}: {m.predict([x_tengah])[0]}")

Jawabannya berpindah-pindah seiring `k` berubah. Inilah pelajaran dari titik ambigu:
**`k` bukan detail teknis, `k` adalah keputusan yang mengubah jawaban.** Untuk titik yang jauh
dari batas, `k` hampir tidak berpengaruh. Untuk titik di dekat batas keputusan, `k` menentukan
segalanya.

## 1.9 Bonus — kalau label ikut menempel di dalam record

Data dunia nyata sering datang dengan label **di dalam** record yang sama, bukan sebagai list
terpisah. Misalnya hasil `df.to_dict("records")` atau satu baris JSON dari API.

Kontrak scikit-learn tetap menuntut `fit(X, y)` dengan dua argumen, jadi kita butuh satu fungsi
pemisah. Sekaligus ini menutup satu jebakan lagi: kalau kolom label ikut terbawa sebagai fitur,
model akan "belajar" dari jawabannya sendiri — bentuk paling telanjang dari **data leakage**.

In [ ]:
records = [
    {"x": 1.0, "y": 1.0, "label": 0},
    {"x": 1.5, "y": 2.0, "label": 0},
    {"x": 2.0, "y": 1.5, "label": 0},
    {"x": 1.2, "y": 1.8, "label": 0},
    {"x": 5.0, "y": 5.0, "label": 1},
    {"x": 5.5, "y": 4.5, "label": 1},
    {"x": 6.0, "y": 5.2, "label": 1},
    {"x": 5.2, "y": 5.8, "label": 1},
]

def pisahkan_label(records, kunci_label="label"):
    '''Pecah list of dict jadi (X tanpa kolom label, y berisi label).'''
    X, y = [], []
    for i, rec in enumerate(records):
        if kunci_label not in rec:
            raise ValueError(f"records[{i}] tidak punya kunci '{kunci_label}'")
        X.append({k: v for k, v in rec.items() if k != kunci_label})
        y.append(rec[kunci_label])
    return X, y


X_dari_records, y_dari_records = pisahkan_label(records)

print("X[0] :", X_dari_records[0])
print("y    :", y_dari_records)
print()

m = KNNClassifierDict(n_neighbors=3).fit(X_dari_records, y_dari_records)
print("feature_names_ :", m.feature_names_, " <- 'label' TIDAK ikut jadi fitur")
print("Prediksi       :", m.predict(X_test))

In [ ]:
# Kalau lupa memisahkan: 'label' ikut jadi fitur, dan model melihat jawabannya
m_bocor = KNNClassifierDict(n_neighbors=3).fit(records, y_dari_records)
print("feature_names_ :", m_bocor.feature_names_, " <- 'label' ikut terbawa!")
print("Akurasi training:", m_bocor.score(records, y_dari_records))
print()
print("Akurasi 1.0 di data training bukan kabar baik di sini --")
print("model cuma menyalin kolom jawaban yang seharusnya tidak pernah ia lihat.")

---
# BAGIAN 2 — Versi `ndarray`

Versi `list of dict` sudah **benar**. Masalahnya ada dua:

1. **Lambat.** Setiap operasi kecil dijalankan interpreter Python satu per satu.
2. **Setiap akses `rec[nama]` adalah lookup hash.** Ini lebih mahal daripada mengambil elemen list
   berdasarkan indeks — jadi versi dict bahkan lebih lambat daripada versi `list of list`.

Numpy memindahkan loop ke kode C terkompilasi. Tapi numpy tidak mengenal dict. Jadi kita perlu
satu **jembatan**: fungsi yang mengubah `list of dict` menjadi matriks — memakai `feature_names_`
sebagai penentu urutan kolom.

Perhatikan bahwa `feature_names_` sekarang punya peran ganda:
di Bagian 1 ia menentukan *apa yang dibandingkan*; di Bagian 2 ia menentukan *kolom ke berapa*.

In [ ]:
import numpy as np

def records_ke_array(records, feature_names):
    '''
    list of dict  ->  ndarray berbentuk (n_record, n_fitur)
    Urutan kolom mengikuti feature_names, bukan urutan kunci di dict.
    '''
    validasi_records(records, feature_names)
    return np.array(
        [[rec[nama] for nama in feature_names] for rec in records],
        dtype=float,
    )


fitur = ["x", "y"]
X_train_arr = records_ke_array(X_train, fitur)
X_test_arr  = records_ke_array(X_test, fitur)
y_train_arr = np.array(y_train)

print("feature_names :", fitur)
print()
print("X_train_arr:\n", X_train_arr)
print("\nshape X_train :", X_train_arr.shape, " -> (n_record, n_fitur)")
print("shape X_test  :", X_test_arr.shape)
print("shape y_train :", y_train_arr.shape)

In [ ]:
# Bukti: urutan kunci diacak, matriksnya tetap identik
X_test_arr_acak = records_ke_array(X_test_acak, fitur)

print("Record asli  :", X_test)
print("Record acak  :", X_test_acak)
print()
print("Matriks dari record asli:\n", X_test_arr)
print("\nMatriks dari record acak:\n", X_test_arr_acak)
print("\nIdentik?", np.array_equal(X_test_arr, X_test_arr_acak))

Inilah inti Bagian 2: begitu data melewati `records_ke_array`, seluruh masalah urutan kunci
**hilang untuk selamanya**. Setelah titik ini, kita berada di dunia numpy biasa —
kolom 0 selalu `"x"`, kolom 1 selalu `"y"`.

Fungsi konversi itu adalah satu-satunya tempat di mana nama fitur masih penting.
Menjaga batas itu tetap sempit dan jelas adalah keputusan desain yang disengaja.

## 2.1 Jarak, tanpa loop

Setelah data jadi matriks, kita bisa memakai **broadcasting**. Saat kita menulis
`X_train_arr - x`, numpy melihat bentuk `(8, 2)` dan `(2,)`, lalu otomatis "merentangkan" `x`
menjadi 8 salinan agar bentuknya cocok — tanpa benar-benar menyalin memorinya.

In [ ]:
x = X_test_arr[0]        # shape (2,)

selisih  = X_train_arr - x            # (8,2) - (2,) -> broadcasting -> (8,2)
kuadrat  = selisih ** 2               # (8,2)
jumlah   = kuadrat.sum(axis=1)        # (8,)  <- axis=1: jumlahkan sepanjang fitur
jarak_np = np.sqrt(jumlah)            # (8,)

print("selisih.shape :", selisih.shape)
print("jumlah.shape  :", jumlah.shape)
print("jarak_np      :", np.round(jarak_np, 4))
print("\nversi dict    :", [round(d, 4) for d in jarak])
print("Sama persis?  :", np.allclose(jarak_np, jarak))

`axis=1` adalah bagian yang paling sering bikin bingung. Cara mengingatnya:
**`axis` menyebutkan sumbu yang akan _dilenyapkan_.** Bentuk `(8, 2)` dijumlahkan dengan `axis=1`
melenyapkan sumbu ke-1 (panjangnya 2, yaitu fitur) dan menyisakan `(8,)` — satu jarak per titik
training. Persis yang kita mau.

Seluruh rangkaian di atas bisa dipadatkan:

In [ ]:
jarak_np  = np.sqrt(((X_train_arr - x) ** 2).sum(axis=1))
jarak_np2 = np.linalg.norm(X_train_arr - x, axis=1)      # fungsi bawaan numpy

print(np.round(jarak_np, 4))
print(np.round(jarak_np2, 4))
print("Identik?", np.allclose(jarak_np, jarak_np2))

## 2.2 `argsort` dan voting, versi numpy

In [ ]:
idx_np = np.argsort(jarak_np)[:3]

print("np.argsort penuh :", np.argsort(jarak_np))
print("Ambil 3 pertama  :", idx_np)
print("Versi dict tadi  :", idx)
print()
print("Label tetangganya:", y_train_arr[idx_np], " <- fancy indexing")

`y_train_arr[idx_np]` disebut **fancy indexing**: mengindeks array dengan array indeks lain.
Di Bagian 1, hal yang sama butuh list comprehension `[y_train[i] for i in idx]`.

> **Optimasi opsional: `np.argpartition`.** `argsort` mengurutkan **seluruh** $n$ jarak, padahal
> kita cuma butuh $k$ terkecil. `np.argpartition(jarak, k)[:k]` hanya memastikan $k$ elemen
> terkecil ada di depan, $O(n)$ alih-alih $O(n \log n)$. Konsekuensinya: $k$ tetangga itu
> **tidak terurut** — tidak masalah untuk voting biasa, bermasalah kalau mau memberi bobot
> berdasarkan peringkat kedekatan.

In [ ]:
classes = np.unique(y_train_arr)
label_tetangga_np = y_train_arr[idx_np]

# Bandingkan tiap label tetangga dengan tiap kelas -> matriks boolean -> jumlahkan
cocok = (label_tetangga_np[:, None] == classes[None, :])
suara = cocok.sum(axis=0)

print("Kelas          :", classes)
print("Label tetangga :", label_tetangga_np)
print("\nMatriks kecocokan (baris=tetangga, kolom=kelas):\n", cocok.astype(int))
print("\nJumlah suara   :", suara)
print("Pemenang       :", classes[suara.argmax()])

Kenapa tidak `np.bincount` saja, yang lebih ringkas? Karena `bincount` menuntut label berupa
**integer non-negatif yang rapat** mulai dari 0. Ia langsung gagal untuk label string
(`"kiri"`, `"kanan"`) atau angka yang melompat (`[10, 20, 30]`). Pendekatan
`np.unique` + perbandingan di atas menerima label apa pun — sama seperti `KNeighborsClassifier` asli.

> **Aturan seri, versi numpy.** `argmax` selalu mengembalikan indeks maksimum **pertama**.
> Karena `np.unique` mengembalikan kelas dalam keadaan **sudah terurut**, "yang pertama" otomatis
> berarti "label terkecil" — persis aturan yang kita tetapkan di Bagian 1. Kedua implementasi
> karena itu akan selalu sepakat, bahkan saat seri. Kita buktikan di Bagian 3.

In [ ]:
suara_seri = np.array([2, 2, 1])          # kelas 0 dan 1 sama-sama 2 suara
print("Suara     :", suara_seri)
print("argmax    :", suara_seri.argmax(), "-> indeks maksimum pertama")
print("np.unique :", np.unique([2, 0, 1, 0]), "-> selalu terurut")

## 2.3 Lompatan besar — semua record test sekaligus

Sampai sini kita masih memproses **satu** record test. Numpy bisa memproses **seluruhnya**
dalam satu operasi, dengan menambahkan sumbu baru memakai `None` (alias `np.newaxis`):

```
X_test [:, None, :]   ->  (m, 1, d)
X_train[None, :, :]   ->  (1, n, d)
                          ---------  broadcasting
selisih               ->  (m, n, d)
```

Setiap `selisih[i, j, :]` adalah vektor selisih antara record test ke-`i` dan record training
ke-`j`. Jumlahkan kuadratnya sepanjang `axis=2`, dan kita dapat **matriks jarak** `(m, n)`.

In [ ]:
selisih_3d = X_test_arr[:, None, :] - X_train_arr[None, :, :]
print("Bentuk selisih 3D   :", selisih_3d.shape, " -> (n_test, n_train, n_fitur)")

D = np.sqrt((selisih_3d ** 2).sum(axis=2))
print("Bentuk matriks jarak:", D.shape, " -> (n_test, n_train)\n")
print("Matriks jarak (baris = record test, kolom = record training):")
print(np.round(D, 3))

Baca matriks itu sebagai tabel: baris ke-0 adalah jarak record test pertama ke kedelapan record
training — empat angka kecil, empat besar. Baris ke-2 (titik tengah) angkanya relatif seragam,
dan itulah bentuk numerik dari "ambigu".

> **Peringatan memori.** Array `selisih_3d` berukuran $m \times n \times d$. Untuk 10.000 record
> test, 10.000 training, dan 50 fitur `float64`, itu **40 GB**. Solusi standarnya memproses data
> test per *batch*. Vektorisasi punya harga, dan harganya adalah RAM.

In [ ]:
k = 3
idx_semua = np.argsort(D, axis=1)[:, :k]        # (n_test, k)
label_semua = y_train_arr[idx_semua]            # (n_test, k)
suara_semua = (label_semua[:, :, None] == classes[None, None, :]).sum(axis=1)

print("Indeks k tetangga terdekat:\n", idx_semua)
print("\nLabel para tetangga:\n", label_semua)
print("\nJumlah suara per kelas:\n", suara_semua)
print("\nPrediksi:", classes[suara_semua.argmax(axis=1)])

## 2.4 Rakit jadi class

Struktur class-nya **tidak berubah sama sekali** — `__init__`, `fit`, `predict`, `score`.
Yang berganti hanya isi mesinnya, ditambah satu langkah konversi di pintu masuk.

Dua method tambahan yang sekarang jadi mudah:

- `kneighbors(X)` — jarak dan indeks tetangga, seperti method senama di `KNeighborsClassifier`.
- `predict_proba(X)` — proporsi suara tiap kelas.

In [ ]:
class KNNClassifierDictArray:
    '''
    K-Nearest Neighbors untuk data list of dict, mesin numpy.

    Input  : list of dict
    Output : np.ndarray
    '''

    def __init__(self, n_neighbors=3):
        self.n_neighbors = n_neighbors

    # ---------- FIT ----------
    def fit(self, X, y):
        if len(X) != len(y):
            raise ValueError(f"X ({len(X)} record) dan y ({len(y)} label) tidak sama panjang")
        if self.n_neighbors > len(X):
            raise ValueError(
                f"n_neighbors={self.n_neighbors} lebih besar dari jumlah data training ({len(X)})"
            )
        if len(X) == 0:
            raise ValueError("X kosong")

        # Belajar urutan fitur kanonik
        kunci = set()
        for rec in X:
            if not isinstance(rec, dict):
                raise TypeError(f"Semua elemen X harus dict, dapat {type(rec).__name__}")
            kunci.update(rec.keys())
        self.feature_names_ = sorted(kunci)
        self.n_features_in_ = len(self.feature_names_)

        # Konversi sekali di sini, lalu hidup di dunia numpy seterusnya
        self.X_train_ = records_ke_array(X, self.feature_names_)
        self.y_train_ = np.asarray(y)
        self.classes_ = np.unique(self.y_train_)

        return self

    # ---------- HELPER ----------
    def _cek_sudah_fit(self):
        if not hasattr(self, "X_train_"):
            raise RuntimeError("Model belum di-fit. Panggil .fit(X, y) terlebih dahulu.")

    def _matriks_jarak(self, X):
        '''Matriks jarak berbentuk (n_test, n_train).'''
        selisih = X[:, None, :] - self.X_train_[None, :, :]
        return np.sqrt((selisih ** 2).sum(axis=2))

    # ---------- KNEIGHBORS ----------
    def kneighbors(self, X):
        '''Kembalikan (jarak, indeks) dari k tetangga terdekat, terurut.'''
        self._cek_sudah_fit()
        X_arr = records_ke_array(X, self.feature_names_)
        D = self._matriks_jarak(X_arr)
        idx = np.argsort(D, axis=1)[:, :self.n_neighbors]
        jarak = np.take_along_axis(D, idx, axis=1)
        return jarak, idx

    def _hitung_suara(self, X):
        _, idx = self.kneighbors(X)
        label_tetangga = self.y_train_[idx]                     # (m, k)
        return (label_tetangga[:, :, None] == self.classes_[None, None, :]).sum(axis=1)

    # ---------- PREDICT ----------
    def predict(self, X):
        suara = self._hitung_suara(X)
        return self.classes_[suara.argmax(axis=1)]              # output: ndarray

    def predict_proba(self, X):
        return self._hitung_suara(X) / self.n_neighbors

    # ---------- SCORE ----------
    def score(self, X, y):
        return float((self.predict(X) == np.asarray(y)).mean())

In [ ]:
model_np = KNNClassifierDictArray(n_neighbors=3).fit(X_train, y_train)

print("feature_names_ :", model_np.feature_names_)
print("classes_       :", model_np.classes_)
print()

prediksi_np = model_np.predict(X_test)
print("Prediksi    :", prediksi_np)
print("Tipe output :", type(prediksi_np))
print("dtype       :", prediksi_np.dtype)
print()
print("Prediksi versi dict :", prediksi)
print("Hasilnya identik?   :", list(prediksi_np) == prediksi)
print("Tahan urutan acak?  :", np.array_equal(model_np.predict(X_test_acak), prediksi_np))

In [ ]:
jarak_k, idx_k = model_np.kneighbors(X_test)
print("Jarak ke k tetangga terdekat:\n", np.round(jarak_k, 4))
print("\nIndeksnya:\n", idx_k)
print("\nproba (kolom mengikuti classes_ =", model_np.classes_, "):")
print(model_np.predict_proba(X_test))

Lihat baris terakhir `predict_proba`: `[0.333, 0.667]`. Itu bukan "model 67% yakin" dalam arti
statistik. Itu artinya **2 dari 3 tetangga** memilih kelas 1. Nilainya hanya bisa
0, 1/3, 2/3, atau 1 — tidak ada nilai lain yang mungkin ketika $k = 3$.

Class ini juga menerima label string, sama seperti versi Bagian 1:

In [ ]:
y_str = ["kiri"] * 4 + ["kanan"] * 4
out_str = KNNClassifierDictArray(n_neighbors=3).fit(X_train, y_str).predict(X_test)
print("Label string :", out_str, "| dtype:", out_str.dtype)

# Validasi tetap jalan, karena records_ke_array memanggil validasi_records
try:
    model_np.predict([{"x": 1.0}])
except ValueError as e:
    print("\nValueError   :", e)

---
# BAGIAN 3 — Verifikasi & benchmark

Tiga pertanyaan penentu:

1. Apakah `feature_names_` kita cocok dengan `DictVectorizer` asli?
2. Apakah prediksinya cocok dengan `KNeighborsClassifier` asli?
3. Seberapa besar bedanya soal kecepatan?

## 3.1 Adu jembatan kita dengan `DictVectorizer`

scikit-learn punya alat resmi untuk mengubah `list of dict` jadi matriks:
`sklearn.feature_extraction.DictVectorizer`. Mari kita pastikan `records_ke_array` kita
berperilaku sama.

In [ ]:
from sklearn.feature_extraction import DictVectorizer

dv = DictVectorizer(sparse=False)
M_sk = dv.fit_transform(X_train)
M_kita = records_ke_array(X_train, sorted({k for r in X_train for k in r}))

print("Fitur menurut DictVectorizer :", list(dv.get_feature_names_out()))
print("Fitur menurut kita           :", model_np.feature_names_)
print("Urutan fitur sama?           :", list(dv.get_feature_names_out()) == model_np.feature_names_)
print()
print("Matriks identik?             :", np.allclose(M_sk, M_kita))

Urutan fiturnya sama, matriksnya sama. Dugaan kita di Bagian 1 terbukti: `DictVectorizer`
memang mengurutkan nama fitur secara alfabetis, jadi `sorted()` adalah pilihan yang tepat.

Tapi ada satu perilaku yang **sengaja kita bedakan** — dan ini penting:

In [ ]:
# Record kehilangan fitur "y"
rusak = [{"x": 1.0}]

print("DictVectorizer :", dv.transform(rusak), " <- 'y' diam-diam diisi 0.0")

try:
    records_ke_array(rusak, model_np.feature_names_)
except ValueError as e:
    print("Punya kita     : ValueError ->", e)

`DictVectorizer` mengisi fitur yang hilang dengan `0.0` tanpa berkata apa-apa. Untuk kasus
penggunaan aslinya — vektorisasi teks, di mana "kata ini tidak muncul" memang berarti nol —
itu perilaku yang benar.

Untuk KNN dengan fitur numerik, itu bencana. Record `{"x": 1.0}` akan diperlakukan sebagai titik
di koordinat `(1.0, 0.0)`, lalu ikut menghitung jarak seolah-olah itu memang lokasinya.
Modelnya tetap jalan, tetap memberi jawaban, dan tetap salah — **tanpa satu pun peringatan**.

Ini pola yang sama untuk ketiga kalinya di notebook ini: urutan kunci, pemutus seri, dan sekarang
nilai yang hilang. Ketiganya adalah **keputusan diam-diam yang diambil oleh library atau bahasa
ketika kita lupa mengambilnya sendiri.** Pesannya bukan "`DictVectorizer` itu buruk" —
melainkan bahwa setiap default punya kasus penggunaan yang diasumsikan, dan tugas kita
memastikan asumsi itu memang cocok dengan masalah kita.

## 3.2 Adu dengan `KNeighborsClassifier` asli

Kita pakai Iris — 150 sampel, 4 fitur, 3 kelas — dan mengubahnya jadi `list of dict`
dengan nama fitur asli, persis bentuk data yang datang dari API atau `df.to_dict("records")`.

Catatan: kita **tidak** melakukan scaling demi menjaga fokus notebook. Di dunia nyata KNN
**wajib** di-scale (`StandardScaler`), karena fitur bersatuan besar akan mendominasi jarak.
Fitur Iris kebetulan sudah sebanding, jadi kita aman untuk saat ini.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

iris = load_iris()
Xtr, Xte, ytr, yte = train_test_split(
    iris.data, iris.target, test_size=0.3, random_state=42, stratify=iris.target
)

def array_ke_records(arr, nama_fitur):
    '''ndarray -> list of dict. Kebalikan dari records_ke_array.'''
    return [dict(zip(nama_fitur, baris)) for baris in arr]


nama = iris.feature_names
Xtr_dict = array_ke_records(Xtr, nama)
Xte_dict = array_ke_records(Xte, nama)

print("Nama fitur:", nama)
print("\nSatu record:")
for k, v in Xtr_dict[0].items():
    print(f"  {k:<22}: {v}")
print("\nTraining:", len(Xtr_dict), "record | Test:", len(Xte_dict), "record")

In [ ]:
# Acak urutan kunci SETIAP record test -- ujian terberat untuk feature_names_
Xte_dict_acak = [acak_urutan_kunci(r, seed=i) for i, r in enumerate(Xte_dict)]

print("Urutan kunci record test #0 :", list(Xte_dict[0].keys()))
print("Setelah diacak              :", list(Xte_dict_acak[0].keys()))
print("Isinya masih sama?          :", Xte_dict_acak == Xte_dict)

Iris adalah ujian yang jauh lebih keras daripada data mainan tadi: 4 fitur, nilainya berbeda-beda,
jadi tidak ada simetri yang bisa menyembunyikan bug urutan. Sebelum menguji modelnya,
mari ukur dulu **seberapa besar kerusakan** yang akan ditimbulkan `jarak_naif` di sini.

In [ ]:
fitur_iris = sorted(nama)
acuan_iris = Xtr_dict[0]

salah = 0
selisih_terbesar = 0.0
for r in Xte_dict_acak:
    benar = euclidean_distance(r, acuan_iris, fitur_iris)
    naif  = jarak_naif(r, acuan_iris)
    if abs(benar - naif) > 1e-9:
        salah += 1
        selisih_terbesar = max(selisih_terbesar, abs(benar - naif))

print(f"Jarak yang salah dihitung jarak_naif : {salah} dari {len(Xte_dict_acak)}")
print(f"Selisih terbesar                     : {selisih_terbesar:.4f}")
print()
print("Di data mainan 2D tadi, banyak kasus kebetulan lolos karena simetri.")
print("Makin banyak fitur, makin kecil peluang bug ini bersembunyi --")
print("dan makin besar kerusakannya kalau tidak ditangani.")

In [ ]:
K = 5

# Milik kita, versi list of dict (Python murni)
pred_dict = KNNClassifierDict(n_neighbors=K).fit(Xtr_dict, ytr.tolist()).predict(Xte_dict)

# Milik kita, versi ndarray
model_arr = KNNClassifierDictArray(n_neighbors=K).fit(Xtr_dict, ytr)
pred_arr = model_arr.predict(Xte_dict)
pred_arr_acak = model_arr.predict(Xte_dict_acak)

# scikit-learn asli, langsung dari ndarray
sk = KNeighborsClassifier(n_neighbors=K).fit(Xtr, ytr)
pred_sk = sk.predict(Xte)

print("feature_names_ :", model_arr.feature_names_)
print()
print(f"Akurasi versi list of dict : {np.mean(np.array(pred_dict) == yte):.4f}")
print(f"Akurasi versi ndarray      : {model_arr.score(Xte_dict, yte):.4f}")
print(f"Akurasi scikit-learn       : {sk.score(Xte, yte):.4f}")
print()
print("dict  == ndarray ?          ", np.array_equal(np.array(pred_dict), pred_arr))
print("kita  == sklearn ?          ", np.array_equal(pred_arr, pred_sk))
print("tahan urutan kunci acak ?   ", np.array_equal(pred_arr_acak, pred_arr))
print(f"Prediksi berbeda dari sklearn: {(pred_arr != pred_sk).sum()} dari {len(yte)}")

Perhatikan baris ketiga dari bawah. `feature_names_` kita berupa `sorted(...)`, sedangkan Iris
versi asli memakai urutan kolom `[sepal length, sepal width, petal length, petal width]` — dan
kedua urutan itu **tidak sama**. Kolom-kolom matriks kita tersusun berbeda dari milik sklearn.

Toh hasilnya tetap identik. Itu masuk akal: jarak Euclidean adalah **penjumlahan** atas seluruh
fitur, dan penjumlahan bersifat komutatif. Menukar urutan kolom tidak mengubah jaraknya —
asalkan **kedua** record dibaca dengan urutan yang sama. Persis syarat yang gagal dipenuhi
`jarak_naif` di Bagian 1.

In [ ]:
print("Urutan kolom Iris asli :", nama)
print("Urutan kolom kita      :", model_arr.feature_names_)
print("Sama?                  :", list(nama) == model_arr.feature_names_)
print()

# Buktikan bahwa jaraknya tetap identik meski urutan kolomnya berbeda
jarak_kita, idx_kita = model_arr.kneighbors(Xte_dict[:5])
jarak_sk,   idx_sk   = sk.kneighbors(Xte[:5])

print("Jarak kita   :\n", np.round(jarak_kita, 5))
print("\nJarak sklearn:\n", np.round(jarak_sk, 5))
print("\nJarak identik? ", np.allclose(jarak_kita, jarak_sk))
print("Indeks identik?", np.array_equal(idx_kita, idx_sk))

Jarak identik sampai digit terakhir, meski urutan kolomnya berbeda. Ini bukti terkuat bahwa
implementasi kita benar — bukan kebetulan menghasilkan label yang sama.

> **Kalau nanti ada selisih satu-dua prediksi,** penyebabnya hampir selalu **seri**, dan itu
> perbedaan konvensi, bukan bug. Beda yang sistematis — itu baru bug.

## 3.3 Benchmark

In [ ]:
import time

rng = np.random.default_rng(42)
n_train, n_test, n_fitur = 1000, 200, 8
nama_fitur = [f"fitur_{i}" for i in range(n_fitur)]

Xb_train = rng.normal(size=(n_train, n_fitur))
yb_train = rng.integers(0, 3, size=n_train)
Xb_test  = rng.normal(size=(n_test, n_fitur))

Xb_train_dict = array_ke_records(Xb_train, nama_fitur)
Xb_test_dict  = array_ke_records(Xb_test, nama_fitur)

print(f"Training: {n_train} record | Test: {n_test} record | {n_fitur} fitur")
print(f"Total perhitungan jarak: {n_train * n_test:,}")

In [ ]:
def ukur(fn, ulang=3):
    '''Jalankan fn beberapa kali, ambil waktu tercepat.'''
    catatan = []
    for _ in range(ulang):
        mulai = time.perf_counter()
        hasil = fn()
        catatan.append(time.perf_counter() - mulai)
    return min(catatan), hasil


m_dict = KNNClassifierDict(n_neighbors=5).fit(Xb_train_dict, yb_train.tolist())
m_arr  = KNNClassifierDictArray(n_neighbors=5).fit(Xb_train_dict, yb_train)
m_sk   = KNeighborsClassifier(n_neighbors=5).fit(Xb_train, yb_train)

t_dict, p_dict = ukur(lambda: m_dict.predict(Xb_test_dict))
t_arr,  p_arr  = ukur(lambda: m_arr.predict(Xb_test_dict))
t_sk,   p_sk   = ukur(lambda: m_sk.predict(Xb_test))

print(f"{'Implementasi':<28}{'Waktu (detik)':>16}{'Relatif':>14}")
print("-" * 58)
print(f"{'list of dict (Python)':<28}{t_dict:>16.4f}{'1.0x':>14}")
print(f"{'ndarray (numpy)':<28}{t_arr:>16.4f}{t_dict/t_arr:>13.1f}x")
print(f"{'KNeighborsClassifier':<28}{t_sk:>16.4f}{t_dict/t_sk:>13.1f}x")
print()
print("Prediksi dict == ndarray ?", np.array_equal(np.array(p_dict), p_arr))

### Ujian sesungguhnya untuk aturan seri

Data acak dengan 3 kelas dan $k=5$ adalah ladang ranjau untuk seri. Mari hitung berapa banyak
yang benar-benar terjadi, lalu pastikan aturan pemutus seri kita bekerja.

In [ ]:
_, idx_b = m_arr.kneighbors(Xb_test_dict)
lab_b = m_arr.y_train_[idx_b]
suara_b = (lab_b[:, :, None] == m_arr.classes_[None, None, :]).sum(axis=1)
jumlah_seri = ((suara_b == suara_b.max(axis=1, keepdims=True)).sum(axis=1) > 1).sum()

print(f"Record test dengan suara seri : {jumlah_seri} dari {len(Xb_test_dict)}")
print(f"Prediksi yang berbeda         : {(np.array(p_dict) != p_arr).sum()}")
print()
print("Kalau kita masih memakai Counter.most_common tanpa pemutus seri,")
print("angka kedua tadi akan puluhan, bukan nol.")

Sebagian besar record mengalami seri, namun kedua implementasi tetap **sepakat sepenuhnya**.
Itu hasil dari menetapkan aturan pemutus seri secara eksplisit di kedua sisi.

Soal kecepatan: numpy jauh lebih unggul, dan hasilnya **identik** — bukan trade-off antara
kecepatan dan ketepatan.

Kenapa scikit-learn bisa lebih cepat lagi? Karena ia tidak memakai brute force. Secara default
`KNeighborsClassifier` memakai `algorithm='auto'`, yang memilih **KD-Tree** atau **Ball Tree** —
struktur data yang memangkas kandidat tetangga tanpa menghitung *semua* jarak, sekitar
$O(\log n)$ per query pada dimensi rendah. (Ironisnya, di atas ~20 fitur struktur pohon itu justru
kalah dari brute force akibat *curse of dimensionality*. Karena itulah setelannya `auto`.)

---
## Ringkasan

### `list of dict` vs `ndarray`

| Langkah | Versi `list of dict` | Versi `ndarray` |
|---|---|---|
| Urutan fitur | `feature_names_` dipakai di **setiap** perhitungan jarak | `feature_names_` dipakai **sekali**, saat konversi |
| Akses nilai | `rec[nama]` (lookup hash) | `arr[i, j]` (offset memori) |
| Selisih | loop `for nama in feature_names` | `X_train - x` (broadcasting) |
| Kuadrat & jumlah | `total += selisih ** 2` | `(selisih ** 2).sum(axis=1)` |
| Semua record training | list comprehension | broadcasting `(n, d)` |
| Semua record test | loop di `predict` | broadcasting `(m, n, d)` |
| Cari k terkecil | `sorted(range(n), key=...)` | `np.argsort(D, axis=1)` |
| Ambil label | `[y[i] for i in idx]` | `y[idx]` (fancy indexing) |
| Voting | `Counter(...)` + pemutus seri | `(label[..., None] == classes).sum(axis=1)` |
| Pemutus seri | `min(kandidat)` | `argmax` + `classes_` terurut |
| Fitur hilang | `ValueError` eksplisit | `ValueError` eksplisit (lewat `records_ke_array`) |
| Tipe output | `list` | `np.ndarray` |

### Enam hal yang layak dibawa pulang

1. **Kontrak `fit`/`predict` itu universal.** Kamu baru saja membangunnya dari nol.
   Setiap model scikit-learn mengikuti pola yang sama.
2. **KNN itu `lazy`.** Tidak ada yang "dipelajari" saat `fit` — semua kerja terjadi di `predict`.
   Training instan, prediksi mahal. Kebalikan dari hampir semua model lain.
3. **Dict tidak punya urutan yang bermakna, jadi jangan pernah memakai `.values()`.**
   Tetapkan `feature_names_` saat `fit`, lalu akses selalu berdasarkan nama.
   `zip(a.values(), b.values())` adalah bug yang menunggu waktu.
4. **`fit` pada data dict punya satu tugas ekstra:** menetapkan urutan fitur kanonik.
   Pada `list of list`, urutan itu gratis dari posisi kolom. Pada dict, kita yang harus memilihnya.
5. **Vektorisasi mengubah eksekusi, bukan logika.** Rumusnya tidak berubah sedikit pun —
   yang berubah hanya siapa yang menjalankan loop. Harganya: memori, lewat matriks `(m, n, d)`.
6. **Diam adalah mode kegagalan terburuk.** Urutan kunci yang salah, seri yang diputus sembarangan,
   fitur hilang yang diisi 0 — ketiganya menghasilkan angka yang tampak wajar, tanpa satu pun
   peringatan. Kalau kamu tidak menetapkan aturannya, bahasa atau library akan menetapkannya
   untukmu, diam-diam.

---

## Latihan

**1. Fitur tak lengkap yang memang wajar.** Data dunia nyata sering benar-benar kehilangan nilai.
Tambahkan parameter `strategi_hilang` pada `fit`, dengan pilihan `"error"` (perilaku sekarang)
dan `"median"` (isi dengan median fitur tersebut **dari data training**).
*Pertanyaan yang lebih penting:* kenapa median harus dihitung dari data training saja,
tidak boleh dari data test? Kaitkan jawabanmu dengan sel kebocoran label di Bagian 1.9.

**2. Fitur kategorikal.** Bagaimana kalau salah satu kunci berisi string, misalnya
`{"x": 1.0, "warna": "merah"}`? `records_ke_array` akan langsung gagal.
Terapkan one-hot encoding di dalam konversi, sehingga `"warna"` menjadi beberapa kolom
`warna=merah`, `warna=biru`. *Petunjuk:* `DictVectorizer` sudah melakukan ini — periksa
`get_feature_names_out()` pada data yang mengandung string, lalu tiru konvensi penamaannya.

**3. Jarak Manhattan.** Tambahkan parameter `metric` yang menerima `"euclidean"` atau
`"manhattan"` ($\sum |p_i - q_i|$). Bandingkan prediksinya pada Iris. Di titik seperti apa
perbedaannya muncul?

**4. Voting berbobot.** `KNeighborsClassifier` punya `weights="distance"`, yang memberi bobot
$1/d$ pada tiap tetangga alih-alih satu suara sama rata. Implementasikan.
*Petunjuk:* hati-hati saat $d = 0$ — apa yang harus terjadi jika record test berimpit persis
dengan record training?

**5. Prediksi per-batch.** Ubah `predict` agar memproses data test dalam potongan 100 record,
supaya matriks `(m, n, d)` tidak pernah dibuat utuh. Verifikasi hasilnya tetap identik.

**6. Kurva `k`.** Plot akurasi test terhadap $k = 1, 3, 5, \dots, 49$ pada Iris.
Di mana titik terbaiknya? Apa yang terjadi saat $k$ mendekati jumlah data training, dan kenapa?